# Source the budget data

*Data Analysis in Python · Topic 16.1*

Your CMO wants to know which channels should get more budget next quarter, and how sure you are. Build the table that answers it, for the whole store: spend and revenue per channel group and day, with the store's organic sessions as the control for demand. Then check it before you model it.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell and the cell that builds the clean `sessions` table (lesson 3.'s steps).
2. In the first cell marked ✏️, create the `spend` table from the three spend files, with lesson 5.'s steps: one row per date and campaign, with the columns `date`, `campaign` and `spend`, the campaign names as `campaign_key()` returns them, and the spend in dollars.
3. In the second cell marked ✏️, create the `campaign_days` table: the `spend` table with each campaign day's revenue from the `sessions` table, joined on the date and the campaign. Add `channel_group`: the third part of the campaign's name, looked up in `CHANNEL_GROUPS`.
4. In the third cell marked ✏️, create the `channel_days` table: one row per channel group and date, with the columns `channel_group`, `date`, `spend`, `revenue` and `organic`, the day's sessions whose `session_medium` is `organic`.
5. In the last cell marked ✏️, create `checks`: a dictionary of named checks, each `True` or `False`:
   - every campaign in `campaign_days` has a channel group
   - every campaign day has a revenue value
   - `channel_days` has a row for every channel group in `CHANNEL_GROUPS` on every day from `START` to `END`

   Then stop the notebook with `assert` if a check fails.
6. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `spend` table has 13,862 rows and \$571,200.03 of spend. The `channel_days` table has 2,924 rows: 4 channel groups × 731 days, with \$1,216,642.52 of revenue. Every check in `checks` is `True`.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
# Every month of 2024 and 2025 as text, "2024-01" to "2025-12": :02d writes the month with two digits
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 13)]
START, END = "2024-01-01", "2025-12-31"
# The media plan's budget lines, by the third part of a campaign's name: Facebook and Instagram share one
CHANNEL_GROUPS = {"search": "Paid Search", "display": "Display", "facebook": "Paid Social", "instagram": "Paid Social", "youtube": "Paid Video"}

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


def campaign_key(names):
    """Return campaign names stripped of spaces and lowercased, so they match across systems."""
    return names.str.strip().str.lower()


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

## The spend side

In [ ]:
google = pd.read_csv(DATA_URL + "google_ads_spend.csv", parse_dates=["date"])
meta = pd.read_csv(DATA_URL + "meta_ads_spend.csv", parse_dates=["Reporting starts"])
youtube = pd.read_csv(DATA_URL + "youtube_spend.csv", parse_dates=["Day"])
google["spend"] = google["cost_micros"] / 1_000_000
spend = pd.concat([
    google.rename(columns={"campaign_name": "campaign"})[["date", "campaign", "spend"]],
    meta.rename(columns={"Reporting starts": "date", "Campaign name": "campaign", "Amount spent (USD)": "spend"})[["date", "campaign", "spend"]],
    youtube.rename(columns={"Day": "date", "Campaign": "campaign", "Cost": "spend"})[["date", "campaign", "spend"]],
], ignore_index=True)
spend["campaign"] = campaign_key(spend["campaign"])
spend = spend.groupby(["date", "campaign"], as_index=False)["spend"].sum()
spend.shape, spend["spend"].sum()

## Spend meets revenue

In [ ]:
revenue = sessions.groupby(["date", "session_campaign"], as_index=False)["revenue"].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})
revenue["campaign"] = campaign_key(revenue["campaign"])
campaign_days = spend.merge(revenue, on=["date", "campaign"], how="left", validate="one_to_one")
campaign_days["channel_group"] = campaign_days["campaign"].str.split("_").str[2].map(CHANNEL_GROUPS)
campaign_days.shape

## One row per channel group and day

In [ ]:
organic = sessions[sessions["session_medium"] == "organic"].groupby("date", as_index=False)["sessions"].sum()
organic = organic.rename(columns={"sessions": "organic"})
channel_days = campaign_days.groupby(["channel_group", "date"], as_index=False)[["spend", "revenue"]].sum()
channel_days = channel_days.merge(organic, on="date", how="left", validate="many_to_one")
channel_days.shape

## Check before you model

In [ ]:
checks = {
    "every campaign is in a channel group": bool(campaign_days["channel_group"].notna().all()),
    "every campaign day has a revenue value": bool(campaign_days["revenue"].notna().all()),
    "every channel group has every day": len(channel_days) == len(set(CHANNEL_GROUPS.values())) * len(pd.date_range(START, END)),
}
assert all(checks.values()), checks
checks

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    spend = found.get("spend")
    if not isinstance(spend, pd.DataFrame) or not {"date", "campaign", "spend"} <= set(spend.columns):
        results.append((False, "spend should be a DataFrame with the columns date, campaign and spend: one row per date and campaign, from the three spend files."))
    elif spend["spend"].sum() > 1_000_000_000:
        results.append((False, f"The spend table's spend adds up to ${spend['spend'].sum():,.0f}: Google Ads' cost_micros are millionths of a dollar. Divide them by 1,000,000, as in topic 5.1."))
    elif len(spend) == 21536:
        results.append((False, "The spend table has 21,536 rows: Meta's file has a row per ad set. Total the spend per date and campaign, as in topic 5.4."))
    elif (spend["campaign"] != spend["campaign"].str.strip().str.lower()).any():
        results.append((False, "Some campaign names in the spend table have spaces or capital letters. Use campaign_key() on them, as in topic 5.2."))
    elif not pd.api.types.is_datetime64_any_dtype(spend["date"]):
        results.append((False, f"The spend table's date column is {spend['date'].dtype}. Read each file's date column as dates with parse_dates=."))
    elif len(spend) != 13862 or abs(spend["spend"].sum() - 571200.03) > 0.01:
        results.append((False, f"The spend table should have 13,862 rows, one per date and campaign, and $571,200.03 of spend. It has {len(spend):,} rows and ${spend['spend'].sum():,.2f}."))
    else:
        results.append((True, "13,862 campaign days and $571,200.03 of spend, from the three platforms' files."))

    campaign_days = found.get("campaign_days")
    if not isinstance(campaign_days, pd.DataFrame) or not {"date", "campaign", "spend", "revenue", "channel_group"} <= set(campaign_days.columns):
        results.append((False, "campaign_days should be a DataFrame with the columns date, campaign, spend, revenue and channel_group."))
    elif len(campaign_days) == 21536:
        results.append((False, "campaign_days has 21,536 rows: Meta's ad-set rows repeat each campaign day's revenue. Total the spend per date and campaign before you join, and add validate=\"one_to_one\" to the merge."))
    elif len(campaign_days) != 13862:
        results.append((False, f"campaign_days should have one row per campaign day in the spend table, 13,862. It has {len(campaign_days):,}: join the revenue to the spend table with how=\"left\"."))
    elif campaign_days["revenue"].isna().any():
        results.append((False, f"{campaign_days['revenue'].isna().sum():,} campaign days have no revenue: their names don't match the sessions table's. Use campaign_key() on the sessions table's campaigns too."))
    elif abs(campaign_days["revenue"].sum() - 1216642.52) > 0.01:
        results.append((False, f"campaign_days's revenue adds up to ${campaign_days['revenue'].sum():,.2f}. The campaign days' revenue from the clean sessions table adds up to $1,216,642.52."))
    elif campaign_days["channel_group"].isna().any() or set(campaign_days["channel_group"]) != {"Paid Search", "Display", "Paid Social", "Paid Video"}:
        results.append((False, f"channel_group should be one of the four budget lines in CHANNEL_GROUPS for every campaign. It has {sorted(map(str, campaign_days['channel_group'].unique()))}: look up the third part of the campaign's name, campaign_days[\"campaign\"].str.split(\"_\").str[2].map(CHANNEL_GROUPS)."))
    else:
        results.append((True, "Every campaign day has its revenue and its channel group: $1,216,642.52 of revenue."))

    channel_days = found.get("channel_days")
    expected = {"Display": (50399.85, 81125.06), "Paid Search": (307200.09, 732994.64), "Paid Social": (141600.04, 285942.00), "Paid Video": (72000.05, 116580.82)}
    if not isinstance(channel_days, pd.DataFrame) or not {"channel_group", "date", "spend", "revenue", "organic"} <= set(channel_days.columns):
        results.append((False, "channel_days should be a DataFrame with the columns channel_group, date, spend, revenue and organic: one row per channel group and date. Use as_index=False in the groupby, so channel_group and date stay columns."))
    elif len(channel_days) == 3655:
        results.append((False, "channel_days has 3,655 rows: five channel groups. Facebook and Instagram are one budget line, Paid Social, in CHANNEL_GROUPS."))
    elif len(channel_days) != 2924:
        results.append((False, f"channel_days should have 2,924 rows: 4 channel groups × 731 days. It has {len(channel_days):,}."))
    elif channel_days["organic"].sum() == 4 * 2506005:
        results.append((False, "The organic column has every session of the day. Paid sessions come from the spend itself, so keep the sessions whose session_medium is organic: they follow demand without any spend behind them."))
    elif channel_days["organic"].sum() != 4 * 602514:
        results.append((False, f"The organic column should have each day's sessions whose session_medium is organic: 602,514 over the two years, on each channel group's rows. It adds up to {channel_days['organic'].sum():,} on all rows."))
    else:
        totals = channel_days.groupby("channel_group")[["spend", "revenue"]].sum()
        wrong = [group for group, (spend_total, revenue_total) in expected.items()
                 if group not in totals.index or abs(totals.loc[group, "spend"] - spend_total) > 0.01 or abs(totals.loc[group, "revenue"] - revenue_total) > 0.01]
        if wrong:
            results.append((False, f"{', '.join(wrong)} should have ${expected[wrong[0]][0]:,.2f} of spend and ${expected[wrong[0]][1]:,.2f} of revenue{' each' if len(wrong) > 1 else ''}. Total campaign_days's spend and revenue per channel group and date."))
        else:
            results.append((True, "2,924 rows: 4 channel groups × 731 days, with each day's organic sessions. Paid Search has $307,200.09 of the $571,200.03."))

    checks = found.get("checks")
    asserted = isinstance(inputs, list) and any(line.strip().startswith("assert ") and "checks" in line for source in inputs[1:-1] for line in source.splitlines())
    if not isinstance(checks, dict) or not checks:
        results.append((False, "checks should be a dictionary of named checks, each True or False."))
    elif len(checks) < 3:
        results.append((False, f"checks has {len(checks)} check{'s' if len(checks) != 1 else ''}. The brief asks for three: channel groups, revenue and a row for every channel group and day."))
    elif not all(isinstance(value, bool) for value in checks.values()):
        results.append((False, f"Each check should be True or False: wrap each one in bool(), as in topic 3.5. checks is {checks}."))
    elif not all(checks.values()):
        results.append((False, f"Every check should pass on this data: {checks}. Read the failing check's code."))
    elif not asserted:
        results.append((False, "Make a failure stop the notebook: assert all(checks.values()), checks."))
    else:
        results.append((True, f"{len(checks)} checks pass, and an assert stops the notebook if one doesn't."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
google = pd.read_csv(DATA_URL + "google_ads_spend.csv", parse_dates=["date"])
meta = pd.read_csv(DATA_URL + "meta_ads_spend.csv", parse_dates=["Reporting starts"])
youtube = pd.read_csv(DATA_URL + "youtube_spend.csv", parse_dates=["Day"])
google["spend"] = google["cost_micros"] / 1_000_000
spend = pd.concat([
    google.rename(columns={"campaign_name": "campaign"})[["date", "campaign", "spend"]],
    meta.rename(columns={"Reporting starts": "date", "Campaign name": "campaign", "Amount spent (USD)": "spend"})[["date", "campaign", "spend"]],
    youtube.rename(columns={"Day": "date", "Campaign": "campaign", "Cost": "spend"})[["date", "campaign", "spend"]],
], ignore_index=True)
spend["campaign"] = campaign_key(spend["campaign"])
spend = spend.groupby(["date", "campaign"], as_index=False)["spend"].sum()
spend.shape, spend["spend"].sum()
```

```python
revenue = sessions.groupby(["date", "session_campaign"], as_index=False)["revenue"].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})
revenue["campaign"] = campaign_key(revenue["campaign"])
campaign_days = spend.merge(revenue, on=["date", "campaign"], how="left", validate="one_to_one")
campaign_days["channel_group"] = campaign_days["campaign"].str.split("_").str[2].map(CHANNEL_GROUPS)
campaign_days.shape
```

```python
organic = sessions[sessions["session_medium"] == "organic"].groupby("date", as_index=False)["sessions"].sum()
organic = organic.rename(columns={"sessions": "organic"})
channel_days = campaign_days.groupby(["channel_group", "date"], as_index=False)[["spend", "revenue"]].sum()
channel_days = channel_days.merge(organic, on="date", how="left", validate="many_to_one")
channel_days.shape
```

```python
checks = {
    "every campaign is in a channel group": bool(campaign_days["channel_group"].notna().all()),
    "every campaign day has a revenue value": bool(campaign_days["revenue"].notna().all()),
    "every channel group has every day": len(channel_days) == len(set(CHANNEL_GROUPS.values())) * len(pd.date_range(START, END)),
}
assert all(checks.values()), checks
checks
```

</details>